# 9. Regression Basics

**Statistical Foundations for Data Science — Notebook 9 of 12**

Correlation (Notebook 5) told us *whether* two variables move together. **Regression** goes
further: it fits an equation, so you can say *how much* $y$ changes per unit of $x$, and
predict $y$ for a new $x$.

This notebook takes the **statistical** view of regression: where the coefficients come
from, what they mean, how uncertain they are, and which assumptions have to hold for those
answers to be trustworthy. Notebook 7 of the next module takes the **machine-learning**
view (pipelines, regularisation, cross-validated model selection).

### What you will learn

1. Simple linear regression and the **least squares** criterion
2. Deriving the slope and intercept, and their link to correlation
3. **Interpreting** coefficients — the sentence you should be able to say out loud
4. $R^2$, adjusted $R^2$, RMSE, MAE: what each one measures
5. **Inference**: standard errors, t-tests and confidence intervals for coefficients
6. **Multiple regression**, and what "controlling for" really means
7. **Categorical predictors** and dummy variables
8. **Polynomial** and interaction terms
9. The four **assumptions** and how to check them with residual plots
10. Outliers, leverage, and influence

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

rng = np.random.default_rng(seed=42)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)

---
## 9.1 The simple linear regression model

We assume the population relationship is

$$y_i = \beta_0 + \beta_1 x_i + \varepsilon_i, \qquad \varepsilon_i \sim N(0, \sigma^2)$$

- $\beta_0$ — **intercept**: the expected $y$ when $x = 0$
- $\beta_1$ — **slope**: the expected change in $y$ for a one-unit increase in $x$
- $\varepsilon_i$ — the **error**: everything about $y$ that $x$ does not explain

We never see $\beta_0, \beta_1$. We estimate them from data as $\hat\beta_0, \hat\beta_1$,
giving fitted values $\hat{y}_i = \hat\beta_0 + \hat\beta_1 x_i$ and **residuals**
$e_i = y_i - \hat{y}_i$.

> **Errors vs residuals.** Errors $\varepsilon_i$ are unobservable deviations from the true
> line. Residuals $e_i$ are observable deviations from the *fitted* line. Everything you
> diagnose in practice, you diagnose with residuals.

In [ ]:
# A dataset we control: advertising spend (thousands) -> sales (thousands of units)
n = 60
ad_spend = rng.uniform(5, 50, n)
TRUE_B0, TRUE_B1, TRUE_SIGMA = 12.0, 1.8, 9.0
sales = TRUE_B0 + TRUE_B1 * ad_spend + rng.normal(0, TRUE_SIGMA, n)

df = pd.DataFrame({"ad_spend": ad_spend, "sales": sales})
print(df.describe().round(2))

plt.scatter(df.ad_spend, df.sales, s=30, color="steelblue")
plt.xlabel("ad spend (thousands)"); plt.ylabel("sales (thousands of units)")
plt.title("Is there a linear relationship?")
plt.show()
print(f"Pearson r = {df.ad_spend.corr(df.sales):.4f}")

---
## 9.2 Least squares: why *this* line?

Infinitely many lines pass through a cloud of points. **Ordinary Least Squares (OLS)**
picks the one that minimises the **sum of squared residuals**:

$$\text{SSE} = \sum_{i=1}^{n}(y_i - \hat\beta_0 - \hat\beta_1 x_i)^2$$

Why squared, not absolute? Three reasons: it has a unique closed-form solution, it is
differentiable everywhere, and under Normal errors it coincides with maximum likelihood.
The cost is **sensitivity to outliers** — squaring makes one bad point very expensive.

Setting the derivatives to zero gives the estimates in closed form:

$$\hat\beta_1 = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sum (x_i - \bar{x})^2}
             = \frac{s_{xy}}{s_x^2} = r\,\frac{s_y}{s_x},
\qquad \hat\beta_0 = \bar{y} - \hat\beta_1 \bar{x}$$

Two facts worth memorising: the slope is just the correlation **rescaled by the ratio of
standard deviations**, and the fitted line always passes through $(\bar{x}, \bar{y})$.

In [ ]:
x, y = df.ad_spend.to_numpy(), df.sales.to_numpy()
xbar, ybar = x.mean(), y.mean()

# Closed-form OLS
b1 = ((x - xbar) * (y - ybar)).sum() / ((x - xbar) ** 2).sum()
b0 = ybar - b1 * xbar

print(f"By hand      : intercept = {b0:.4f}, slope = {b1:.4f}")

# Via correlation and the sd ratio
r = np.corrcoef(x, y)[0, 1]
print(f"r * sy/sx    : slope = {r * y.std(ddof=1) / x.std(ddof=1):.4f}")

# Via scipy
lr = stats.linregress(x, y)
print(f"scipy        : intercept = {lr.intercept:.4f}, slope = {lr.slope:.4f}")

# Via scikit-learn
model = LinearRegression().fit(x.reshape(-1, 1), y)
print(f"scikit-learn : intercept = {model.intercept_:.4f}, slope = {model.coef_[0]:.4f}")
print(f"\nTrue values  : intercept = {TRUE_B0}, slope = {TRUE_B1}")

In [ ]:
# Show that least squares really is the minimum
def sse(b0_try, b1_try):
    return ((y - b0_try - b1_try * x) ** 2).sum()

slopes = np.linspace(b1 - 1.2, b1 + 1.2, 200)
plt.plot(slopes, [sse(b0, s_) for s_ in slopes], lw=2, color="steelblue")
plt.axvline(b1, color="crimson", ls="--", label=f"OLS slope = {b1:.3f}")
plt.xlabel("candidate slope"); plt.ylabel("sum of squared errors")
plt.title("SSE is a parabola; OLS sits at its minimum")
plt.legend(); plt.show()

print("Three candidate lines and their SSE:")
for label, (bb0, bb1) in {"OLS": (b0, b1), "too flat": (b0 + 15, b1 - 0.6),
                          "too steep": (b0 - 15, b1 + 0.6)}.items():
    print(f"  {label:<10} intercept={bb0:7.2f} slope={bb1:5.2f}  SSE = {sse(bb0, bb1):10.1f}")

In [ ]:
fitted = b0 + b1 * x
resid = y - fitted

fig, ax = plt.subplots(1, 2, figsize=(13, 4.4))
ax[0].scatter(x, y, s=30, color="steelblue", zorder=3)
xs = np.linspace(x.min(), x.max(), 100)
ax[0].plot(xs, b0 + b1 * xs, color="crimson", lw=2, label=f"y = {b0:.2f} + {b1:.2f}x")
for xi, yi, fi in zip(x, y, fitted):
    ax[0].plot([xi, xi], [yi, fi], color="grey", lw=0.8, alpha=0.7)
ax[0].scatter([xbar], [ybar], color="black", s=90, marker="X", zorder=4,
              label="(x-bar, y-bar)")
ax[0].set_xlabel("ad spend"); ax[0].set_ylabel("sales")
ax[0].set_title("Fitted line; grey segments are the residuals")
ax[0].legend(fontsize=8)

ax[1].scatter(fitted, resid, s=30, color="steelblue")
ax[1].axhline(0, color="crimson", lw=1.5)
ax[1].set_xlabel("fitted values"); ax[1].set_ylabel("residual")
ax[1].set_title("Residuals vs fitted: should look like structureless noise")
plt.tight_layout(); plt.show()

print(f"Sum of residuals  = {resid.sum():.10f}   (always 0 when an intercept is fitted)")
print(f"Corr(x, residual) = {np.corrcoef(x, resid)[0,1]:.10f}   (always 0)")

---
## 9.3 Interpreting the coefficients

This is where marks are won and lost. For $\widehat{\text{sales}} = \hat\beta_0 + \hat\beta_1 \cdot \text{ad spend}$:

- **Slope:** *"Each additional ₹1,000 of ad spend is associated with an average increase of
  $\hat\beta_1$ thousand units sold."*
- **Intercept:** *"With zero ad spend, expected sales are $\hat\beta_0$ thousand units."*

Four disciplines to keep:

1. Say **"associated with"**, not "causes" — unless the $x$ was randomly assigned
2. Say **"on average"** — the line predicts the mean of $y$, not any individual $y$
3. **Keep the units** — a slope without units is meaningless
4. Do not interpret the intercept when $x = 0$ is outside the data range (**extrapolation**)

In [ ]:
print(f"Fitted model: sales = {b0:.3f} + {b1:.3f} * ad_spend\n")
print("Correct interpretation:")
print(f"  Each extra 1,000 of ad spend is associated with an average increase of")
print(f"  {b1:.3f} thousand units sold ({b1*1000:.0f} units).")
print(f"  Doubling spend from 20 to 40 predicts an extra {b1*20:.1f} thousand units.\n")

print("Prediction at a few spend levels:")
for spend in (10, 25, 40):
    print(f"  spend = {spend:>2} -> predicted sales = {b0 + b1*spend:6.2f}")

print(f"\nData range of ad_spend: {x.min():.1f} to {x.max():.1f}")
print(f"EXTRAPOLATION WARNING: predicting at spend = 200 gives "
      f"{b0 + b1*200:.1f}, which the data cannot support.")

---
## 9.4 How good is the fit?

### The variance decomposition

$$\underbrace{\sum(y_i - \bar{y})^2}_{\text{SST, total}}
= \underbrace{\sum(\hat{y}_i - \bar{y})^2}_{\text{SSR, explained}}
+ \underbrace{\sum(y_i - \hat{y}_i)^2}_{\text{SSE, unexplained}}$$

### $R^2$ — coefficient of determination

$$R^2 = \frac{\text{SSR}}{\text{SST}} = 1 - \frac{\text{SSE}}{\text{SST}}$$

The **proportion of variance in $y$ explained by the model**. For simple regression,
$R^2 = r^2$ exactly.

### Adjusted $R^2$

$R^2$ never decreases when you add a predictor, even a random one. Adjusted $R^2$ charges
rent for each variable:

$$R^2_{adj} = 1 - \frac{\text{SSE}/(n-k-1)}{\text{SST}/(n-1)}$$

### Error metrics in the units of $y$

$$\text{RMSE} = \sqrt{\frac{\text{SSE}}{n}}, \qquad
\text{MAE} = \frac{1}{n}\sum|y_i - \hat{y}_i|$$

RMSE punishes large errors more (it is in the same family as least squares); MAE is
robust and reads as "typical error". Report at least one of them — $R^2$ alone never tells
a stakeholder whether the model is accurate enough to use.

In [ ]:
SST = ((y - ybar) ** 2).sum()
SSR = ((fitted - ybar) ** 2).sum()
SSE = (resid ** 2).sum()

print(f"SST = {SST:10.2f}   (total variation in sales)")
print(f"SSR = {SSR:10.2f}   (explained by ad spend)")
print(f"SSE = {SSE:10.2f}   (left over)")
print(f"SSR + SSE = {SSR + SSE:.2f}  -> matches SST\n")

R2 = 1 - SSE / SST
k = 1
R2_adj = 1 - (SSE / (n - k - 1)) / (SST / (n - 1))
rmse = np.sqrt(SSE / n)
mae = np.abs(resid).mean()

print(f"R^2          = {R2:.4f}   ({R2*100:.1f}% of variance explained)")
print(f"r^2          = {r**2:.4f}   (identical, as it must be for one predictor)")
print(f"Adjusted R^2 = {R2_adj:.4f}")
print(f"RMSE         = {rmse:.4f} thousand units")
print(f"MAE          = {mae:.4f} thousand units")
print(f"\nsklearn check: R2 = {r2_score(y, fitted):.4f}, "
      f"RMSE = {np.sqrt(mean_squared_error(y, fitted)):.4f}, "
      f"MAE = {mean_absolute_error(y, fitted):.4f}")
print(f"\nResidual standard error (sigma-hat) = {np.sqrt(SSE/(n-2)):.3f}  (true sigma = {TRUE_SIGMA})")

In [ ]:
# R^2 always rises when you add junk predictors; adjusted R^2 does not.
X_base = x.reshape(-1, 1)
print(f"{'predictors':>12} {'R^2':>10} {'adjusted R^2':>15}")
for extra in range(0, 16, 3):
    X_aug = np.column_stack([X_base] + [rng.normal(size=n) for _ in range(extra)])
    m = LinearRegression().fit(X_aug, y)
    r2 = m.score(X_aug, y)
    kk = X_aug.shape[1]
    adj = 1 - (1 - r2) * (n - 1) / (n - kk - 1)
    print(f"{kk:>12} {r2:>10.4f} {adj:>15.4f}")
print("\nAll the extra columns are pure noise. R^2 climbs anyway -- never select a model on R^2.")

---
## 9.5 Inference: is the slope real?

The coefficient estimate is a statistic, so it has a sampling distribution. The standard
error of the slope is

$$\operatorname{SE}(\hat\beta_1) = \frac{\hat\sigma}{\sqrt{\sum(x_i-\bar{x})^2}},
\qquad \hat\sigma = \sqrt{\frac{\text{SSE}}{n-2}}$$

Then the usual t-machinery from Notebook 7 applies, with $df = n - k - 1$:

$$H_0: \beta_1 = 0 \qquad t = \frac{\hat\beta_1}{\operatorname{SE}(\hat\beta_1)}
\qquad \text{CI} = \hat\beta_1 \pm t_{\alpha/2,\,df}\operatorname{SE}(\hat\beta_1)$$

Notice what makes the standard error small: **low noise** ($\hat\sigma$), **lots of data**,
and a **wide spread of $x$ values**. That last one is a design insight — if you can choose
where to measure, spread your $x$ out.

In [ ]:
sigma_hat = np.sqrt(SSE / (n - 2))
Sxx = ((x - xbar) ** 2).sum()

se_b1 = sigma_hat / np.sqrt(Sxx)
se_b0 = sigma_hat * np.sqrt(1/n + xbar**2 / Sxx)

t_b1 = b1 / se_b1
t_b0 = b0 / se_b0
dfree = n - 2
tc = stats.t(dfree).ppf(0.975)

summary = pd.DataFrame({
    "coefficient": ["intercept", "ad_spend"],
    "estimate":    [b0, b1],
    "std_error":   [se_b0, se_b1],
    "t":           [t_b0, t_b1],
    "p_value":     [2*stats.t(dfree).sf(abs(t_b0)), 2*stats.t(dfree).sf(abs(t_b1))],
    "ci_low":      [b0 - tc*se_b0, b1 - tc*se_b1],
    "ci_high":     [b0 + tc*se_b0, b1 + tc*se_b1],
})
print(summary.round(4).to_string(index=False))
print(f"\nResidual standard error: {sigma_hat:.3f} on {dfree} degrees of freedom")
print(f"scipy cross-check: slope SE = {lr.stderr:.4f}, p = {lr.pvalue:.3e}")
print(f"\nThe 95% CI for the slope [{b1-tc*se_b1:.3f}, {b1+tc*se_b1:.3f}] excludes 0,")
print(f"and contains the true value {TRUE_B1}.")

In [ ]:
# Confidence band (for the mean of y) vs prediction band (for a new observation)
xs = np.linspace(x.min() - 2, x.max() + 2, 200)
y_hat = b0 + b1 * xs

se_mean = sigma_hat * np.sqrt(1/n + (xs - xbar)**2 / Sxx)
se_pred = sigma_hat * np.sqrt(1 + 1/n + (xs - xbar)**2 / Sxx)

plt.scatter(x, y, s=25, color="steelblue", alpha=0.8, label="data")
plt.plot(xs, y_hat, color="crimson", lw=2, label="fitted line")
plt.fill_between(xs, y_hat - tc*se_mean, y_hat + tc*se_mean,
                 color="crimson", alpha=0.25, label="95% CI for the MEAN")
plt.plot(xs, y_hat - tc*se_pred, "k--", lw=1, label="95% interval for a NEW point")
plt.plot(xs, y_hat + tc*se_pred, "k--", lw=1)
plt.xlabel("ad spend"); plt.ylabel("sales")
plt.title("Confidence band vs prediction band")
plt.legend(fontsize=8); plt.show()

print("The confidence band is narrow: the average line is well pinned down.")
print("The prediction band is much wider: individual outcomes also carry the error term.")
print("Both are narrowest at x-bar and flare out at the edges -- another reason")
print("extrapolation is dangerous.")

---
## 9.6 Multiple regression

With several predictors:

$$y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \dots + \beta_k x_k + \varepsilon$$

In matrix form the OLS solution is the **normal equation**:

$$\hat{\boldsymbol\beta} = (X^\top X)^{-1} X^\top \mathbf{y}$$

**The crucial shift in interpretation.** $\beta_j$ is now the expected change in $y$ for a
one-unit increase in $x_j$ **holding all other predictors fixed**. This is why people say
regression "controls for" variables — and why a coefficient can change sign when you add a
predictor (Simpson's paradox, Notebook 5, in coefficient form).

In [ ]:
# Marketing data with three channels and a confounded "season" effect
m = 200
season = rng.normal(0, 1, m)                       # a latent demand driver
tv     = 20 + 8 * season + rng.normal(0, 5, m)     # we spend more on TV in high season
radio  = 15 + rng.normal(0, 6, m)
online = 30 + 3 * season + rng.normal(0, 7, m)
revenue = (50 + 1.2*tv + 0.6*radio + 2.1*online + 14*season + rng.normal(0, 8, m))

mk = pd.DataFrame({"tv": tv, "radio": radio, "online": online,
                   "season": season, "revenue": revenue})

print("Correlation with revenue:")
print(mk.corr()["revenue"].drop("revenue").round(3).to_string())

In [ ]:
# OLS by the normal equation, then with sklearn
features = ["tv", "radio", "online"]
X = mk[features].to_numpy()
yv = mk["revenue"].to_numpy()

X1 = np.column_stack([np.ones(len(X)), X])                 # add the intercept column
beta = np.linalg.solve(X1.T @ X1, X1.T @ yv)               # solve, don't invert
print("Normal equation :", np.round(beta, 4))

mod = LinearRegression().fit(X, yv)
print("scikit-learn    :", np.round(np.r_[mod.intercept_, mod.coef_], 4))
print(f"\nR^2 = {mod.score(X, yv):.4f}")
print(f"True coefficients: tv=1.2, radio=0.6, online=2.1 (plus a hidden season effect)")

In [ ]:
# A reusable OLS summary table (what statsmodels would print)
def ols_summary(X, y, names):
    '''Full OLS fit with coefficient inference. X excludes the intercept column.'''
    n_, k_ = X.shape
    X1 = np.column_stack([np.ones(n_), X])
    beta = np.linalg.solve(X1.T @ X1, X1.T @ y)
    fit = X1 @ beta
    res = y - fit
    sse = (res ** 2).sum()
    sst = ((y - y.mean()) ** 2).sum()
    dfree = n_ - k_ - 1
    sigma2 = sse / dfree
    cov = sigma2 * np.linalg.inv(X1.T @ X1)
    se = np.sqrt(np.diag(cov))
    tvals = beta / se
    pvals = 2 * stats.t(dfree).sf(np.abs(tvals))
    tcrit = stats.t(dfree).ppf(0.975)
    tab = pd.DataFrame({
        "term": ["intercept"] + list(names),
        "estimate": beta, "std_error": se, "t": tvals, "p_value": pvals,
        "ci_low": beta - tcrit*se, "ci_high": beta + tcrit*se,
    })
    r2 = 1 - sse/sst
    stats_dict = {
        "n": n_, "k": k_, "R2": r2,
        "R2_adj": 1 - (1-r2)*(n_-1)/dfree,
        "RMSE": np.sqrt(sse/n_), "resid_se": np.sqrt(sigma2),
        "F": (sst - sse)/k_ / sigma2,
    }
    stats_dict["F_pvalue"] = stats.f(k_, dfree).sf(stats_dict["F"])
    return tab, stats_dict, res, fit

tab, st, res_mk, fit_mk = ols_summary(X, yv, features)
print(tab.round(4).to_string(index=False))
print()
for kk, vv in st.items():
    print(f"  {kk:<9} = {vv:,.4f}" if isinstance(vv, float) else f"  {kk:<9} = {vv}")
print("\nThe F-test asks whether ALL slopes are zero at once; a tiny p-value means")
print("the model as a whole beats predicting the mean of y.")

In [ ]:
# What "controlling for" does: add the confounder and watch the TV coefficient change
tab_no, st_no, _, _ = ols_summary(mk[["tv"]].to_numpy(), yv, ["tv"])
tab_yes, st_yes, _, _ = ols_summary(mk[["tv", "season"]].to_numpy(), yv, ["tv", "season"])

print("TV alone:")
print(tab_no.round(3).to_string(index=False))
print("\nTV controlling for season:")
print(tab_yes.round(3).to_string(index=False))
print(f"\nTV coefficient: {tab_no.loc[1,'estimate']:.3f} alone vs "
      f"{tab_yes.loc[1,'estimate']:.3f} after controlling (true value 1.2).")
print("Ignoring season credits TV with demand it did not create -- omitted variable bias.")

---
## 9.7 Categorical predictors and dummy variables

A regression needs numbers, so categories become **indicator (dummy) columns**. With $L$
levels you create $L-1$ dummies and leave one out as the **reference level** — including
all $L$ makes the columns perfectly collinear (the *dummy variable trap*), and $X^\top X$
becomes singular.

Each dummy coefficient reads as: *"the average difference in $y$ between this level and the
reference level, holding everything else fixed."*

In [ ]:
store = pd.DataFrame({
    "region": rng.choice(["North", "South", "East"], 180, p=[0.4, 0.35, 0.25]),
    "footfall": rng.uniform(100, 900, 180),
})
region_lift = {"North": 0.0, "South": 25.0, "East": -18.0}     # North is the reference
store["sales"] = (60 + 0.35 * store.footfall
                  + store.region.map(region_lift) + rng.normal(0, 20, 180))

dummies = pd.get_dummies(store["region"], prefix="region", drop_first=True, dtype=float)
print("First rows of the design matrix (North is the dropped reference level):")
print(pd.concat([store[["footfall"]], dummies], axis=1).head())

Xc = pd.concat([store[["footfall"]], dummies], axis=1).to_numpy()
tab_c, st_c, _, _ = ols_summary(Xc, store.sales.to_numpy(),
                                ["footfall"] + list(dummies.columns))
print()
print(tab_c.round(3).to_string(index=False))
print(f"\nR^2 = {st_c['R2']:.4f}")
print("\nReading it: a South store sells about "
      f"{tab_c.loc[2,'estimate']:.1f} more than a North store with the same footfall;")
print(f"an East store about {abs(tab_c.loc[3,'estimate']):.1f} less. True lifts: +25 and -18.")

---
## 9.8 Non-linear relationships: polynomial and interaction terms

"Linear regression" means linear **in the coefficients**, not in the predictors. You are
free to add $x^2$, $\log x$, or $x_1 x_2$ as columns and still use OLS.

- **Polynomial terms** capture curvature: diminishing returns, U-shapes
- **Interaction terms** ($x_1 \times x_2$) let the effect of one variable depend on another
- **Log transforms** turn multiplicative relationships into additive ones, and change the
  interpretation to *percentages* (a log-log slope is an **elasticity**)

The cost is **overfitting**, which is exactly what Notebooks 11 and 12 are about.

In [ ]:
# Diminishing returns: sales rise with spend, then flatten
spend = rng.uniform(0, 100, 150)
sat_sales = 40 * np.log1p(spend) + rng.normal(0, 18, 150)

deg_fits = {}
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
grid = np.linspace(0, 100, 300)
for ax, deg in zip(axes, [1, 2, 6]):
    Xp = np.column_stack([spend ** p for p in range(1, deg + 1)])
    mp = LinearRegression().fit(Xp, sat_sales)
    Gp = np.column_stack([grid ** p for p in range(1, deg + 1)])
    ax.scatter(spend, sat_sales, s=14, alpha=0.6, color="steelblue")
    ax.plot(grid, mp.predict(Gp), color="crimson", lw=2)
    r2d = mp.score(Xp, sat_sales)
    deg_fits[deg] = r2d
    ax.set_title(f"degree {deg}: R^2 = {r2d:.4f}", fontsize=10)
    ax.set_xlabel("spend")
axes[0].set_ylabel("sales")
plt.tight_layout(); plt.show()

# The honest comparison: a log term, which matches how the data was generated
Xlog = np.log1p(spend).reshape(-1, 1)
mlog = LinearRegression().fit(Xlog, sat_sales)
print(f"log(1+spend) model: R^2 = {mlog.score(Xlog, sat_sales):.4f} with ONE predictor")
print(f"degree-6 polynomial: R^2 = {deg_fits[6]:.4f} with SIX predictors")
print("\nThe degree-6 curve wiggles at the edges -- high R^2 bought with instability.")
print("Choosing a transform that matches the mechanism beats throwing powers at it.")

In [ ]:
# Interaction: the effect of discount depends on whether it is a weekend
mk2 = pd.DataFrame({
    "discount": rng.uniform(0, 30, 300),
    "weekend": rng.integers(0, 2, 300),
})
mk2["units"] = (100 + 2.0*mk2.discount + 30*mk2.weekend
                + 1.8*mk2.discount*mk2.weekend + rng.normal(0, 15, 300))

X_no  = mk2[["discount", "weekend"]].to_numpy()
X_int = np.column_stack([X_no, mk2.discount * mk2.weekend])

t_no, s_no, _, _   = ols_summary(X_no,  mk2.units.to_numpy(), ["discount", "weekend"])
t_int, s_int, _, _ = ols_summary(X_int, mk2.units.to_numpy(),
                                 ["discount", "weekend", "discount_x_weekend"])
print(f"Without interaction: R^2 = {s_no['R2']:.4f}")
print(f"With interaction   : R^2 = {s_int['R2']:.4f}\n")
print(t_int.round(3).to_string(index=False))
print("\nReading it: on weekdays each point of discount adds about "
      f"{t_int.loc[1,'estimate']:.2f} units;")
print(f"at weekends it adds {t_int.loc[1,'estimate'] + t_int.loc[3,'estimate']:.2f} units.")

for wknd, colour in [(0, "steelblue"), (1, "crimson")]:
    sub = mk2[mk2.weekend == wknd]
    plt.scatter(sub.discount, sub.units, s=14, alpha=0.6, color=colour,
                label="weekend" if wknd else "weekday")
    cf = np.polyfit(sub.discount, sub.units, 1)
    gx = np.linspace(0, 30, 50)
    plt.plot(gx, np.polyval(cf, gx), color=colour, lw=2)
plt.xlabel("discount (%)"); plt.ylabel("units sold")
plt.title("Different slopes = an interaction")
plt.legend(); plt.show()

---
## 9.9 The assumptions, and how to check them

Remember them as **LINE**:

| | Assumption | How to check | If violated |
|---|---|---|---|
| **L** | **Linearity** — the relationship really is linear in the predictors | residuals vs fitted: no curve | add terms, transform |
| **I** | **Independence** — errors are uncorrelated | study design; residuals vs time/order | time-series or clustered models |
| **N** | **Normality** of the errors | Q–Q plot of residuals | large $n$ is forgiving (CLT); or transform |
| **E** | **Equal variance** (homoscedasticity) | residuals vs fitted: constant band | log $y$, weighted least squares, robust SEs |

Plus: **no perfect multicollinearity** among predictors (check VIF — Notebook 5).

Note what each assumption protects. Linearity and independence affect the **estimates**.
Normality and equal variance mostly affect the **standard errors, p-values and intervals**
— your slope can still be a fine point estimate while its p-value is nonsense.

In [ ]:
def diagnostic_plots(residuals, fitted_vals, title=""):
    '''The standard four-panel regression diagnostic.'''
    fig, ax = plt.subplots(1, 4, figsize=(17, 3.6))

    ax[0].scatter(fitted_vals, residuals, s=16, alpha=0.7, color="steelblue")
    ax[0].axhline(0, color="crimson", lw=1.4)
    lo = np.polyfit(fitted_vals, residuals, 2)
    gx = np.linspace(fitted_vals.min(), fitted_vals.max(), 100)
    ax[0].plot(gx, np.polyval(lo, gx), color="darkorange", lw=1.6)
    ax[0].set_title("Residuals vs fitted\n(want: flat band)", fontsize=9)
    ax[0].set_xlabel("fitted"); ax[0].set_ylabel("residual")

    stats.probplot(residuals, dist="norm", plot=ax[1])
    ax[1].set_title("Q-Q plot of residuals\n(want: straight line)", fontsize=9)

    std_res = residuals / residuals.std(ddof=1)
    ax[2].scatter(fitted_vals, np.sqrt(np.abs(std_res)), s=16, alpha=0.7, color="steelblue")
    ax[2].set_title("Scale-location\n(want: no trend)", fontsize=9)
    ax[2].set_xlabel("fitted"); ax[2].set_ylabel("sqrt|std residual|")

    ax[3].plot(residuals, marker="o", ms=3, lw=0.6, color="steelblue")
    ax[3].axhline(0, color="crimson", lw=1.2)
    ax[3].set_title("Residuals in data order\n(want: no pattern)", fontsize=9)
    ax[3].set_xlabel("observation index")

    if title:
        fig.suptitle(title, y=1.04, fontsize=11)
    plt.tight_layout(); plt.show()

diagnostic_plots(res_mk, fit_mk, "Marketing model: a well-behaved fit")
print(f"Shapiro-Wilk on residuals: p = {stats.shapiro(res_mk).pvalue:.4f}")

In [ ]:
# Now three deliberately broken models, so you learn to recognise the signatures
xb = np.linspace(1, 30, 200)

# (a) non-linearity ignored
y_curve = 3 + 0.4 * xb**2 + rng.normal(0, 15, 200)
f_a = np.polyval(np.polyfit(xb, y_curve, 1), xb)
diagnostic_plots(y_curve - f_a, f_a, "(a) LINEARITY violated: residuals bend in a U")

# (b) heteroscedasticity
y_fan = 10 + 2*xb + rng.normal(0, 1, 200) * xb
f_b = np.polyval(np.polyfit(xb, y_fan, 1), xb)
diagnostic_plots(y_fan - f_b, f_b, "(b) EQUAL VARIANCE violated: the band fans out")

In [ ]:
# (c) dependence: errors drift over time (autocorrelation)
drift = np.cumsum(rng.normal(0, 2.5, 200))
y_dep = 10 + 2*xb + drift
f_c = np.polyval(np.polyfit(xb, y_dep, 1), xb)
res_c = y_dep - f_c
diagnostic_plots(res_c, f_c, "(c) INDEPENDENCE violated: residuals wander in long runs")

# The Durbin-Watson statistic: about 2 means no autocorrelation, near 0 means positive
def durbin_watson(res):
    d = np.diff(res)
    return (d ** 2).sum() / (res ** 2).sum()

print(f"Durbin-Watson, healthy model : {durbin_watson(res_mk):.3f}  (want ~2)")
print(f"Durbin-Watson, drifting model: {durbin_watson(res_c):.3f}  (positive autocorrelation)")
print("\nWith autocorrelated errors the coefficients may be fine but the standard errors")
print("are far too small, so everything looks more significant than it is.")

---
## 9.10 Outliers, leverage and influence

Three different problems, often confused:

- **Outlier** — unusual $y$ for its $x$ (a large residual)
- **High leverage** — unusual $x$, far from $\bar{x}$; it *can* pull the line hard
- **Influential** — actually *does* change the fit. Influence ≈ outlier × leverage.

**Cook's distance** measures influence directly: how much do all fitted values move if you
delete point $i$?

$$D_i = \frac{e_i^2}{(k+1)\hat\sigma^2}\cdot\frac{h_i}{(1-h_i)^2}$$

where $h_i$ is the **hat value** (leverage). Rules of thumb: investigate $h_i > 2(k+1)/n$
or $D_i > 4/n$.

Finding an influential point is not permission to delete it. Investigate: is it a data
error (fix it), a different population (model it separately), or just a rare real event
(keep it and report the sensitivity)?

In [ ]:
def leverage_and_cooks(X, y):
    '''Hat values and Cook's distance for an OLS fit.'''
    n_, k_ = X.shape
    X1 = np.column_stack([np.ones(n_), X])
    XtX_inv = np.linalg.inv(X1.T @ X1)
    H_diag = np.einsum("ij,jk,ik->i", X1, XtX_inv, X1)     # diagonal of the hat matrix
    beta = XtX_inv @ X1.T @ y
    res = y - X1 @ beta
    sigma2 = (res ** 2).sum() / (n_ - k_ - 1)
    cooks = res**2 / ((k_ + 1) * sigma2) * H_diag / (1 - H_diag)**2
    return H_diag, cooks, res

# Take the clean advertising data and add three troublesome points
x_bad = np.r_[x, 27.0, 95.0, 92.0]
y_bad = np.r_[y, 140.0, 190.0, 25.0]
labels = ["outlier only", "high leverage, on-trend", "influential (both)"]

Xb = x_bad.reshape(-1, 1)
h, D, rr = leverage_and_cooks(Xb, y_bad)
n_b = len(x_bad)

print(f"Thresholds: leverage > {2*2/n_b:.4f},  Cook's D > {4/n_b:.4f}\n")
for lab, idx in zip(labels, range(n_b - 3, n_b)):
    print(f"  {lab:<26} residual={rr[idx]:+8.2f}  leverage={h[idx]:.4f}  Cook's D={D[idx]:.4f}")
print(f"\n  typical clean point         residual={rr[0]:+8.2f}  leverage={h[0]:.4f}  Cook's D={D[0]:.4f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.4))

ax[0].scatter(x, y, s=25, color="steelblue", label="original data")
for lab, xi, yi, c in zip(labels, x_bad[-3:], y_bad[-3:], ["darkorange", "seagreen", "crimson"]):
    ax[0].scatter([xi], [yi], s=110, color=c, marker="D", zorder=5, label=lab)
gx = np.linspace(0, 100, 50)
ax[0].plot(gx, b0 + b1*gx, color="black", lw=2, label="fit without the 3 points")
mb = LinearRegression().fit(Xb, y_bad)
ax[0].plot(gx, mb.intercept_ + mb.coef_[0]*gx, color="crimson", ls="--", lw=2,
           label="fit with all points")
ax[0].set_xlabel("ad spend"); ax[0].set_ylabel("sales"); ax[0].legend(fontsize=7)
ax[0].set_title("Three kinds of unusual point")

ax[1].stem(D)
ax[1].axhline(4/n_b, color="crimson", ls="--", label=f"threshold 4/n = {4/n_b:.3f}")
ax[1].set_xlabel("observation index"); ax[1].set_ylabel("Cook's distance")
ax[1].set_title("Cook's distance identifies the culprit")
ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f"Slope without the extra points : {b1:.4f}")
print(f"Slope with them                : {mb.coef_[0]:.4f}")
print("\nThe on-trend high-leverage point is harmless. The one that is BOTH far out in x")
print("and off-trend in y drags the whole line.")

---
## Exercises

**Exercise 1.** Given the data below, fit a simple linear regression **by hand** (no
sklearn): compute $\hat\beta_1$, $\hat\beta_0$, $R^2$, RMSE, the standard error of the
slope, and a 95% CI for the slope. Then verify with `scipy.stats.linregress`.

In [ ]:
# --- Solution 1 -------------------------------------------------------------
hours = np.array([2, 3, 4, 5, 6, 7, 8, 9, 10, 12])
score = np.array([52, 55, 61, 64, 66, 72, 74, 79, 81, 88])

nn = len(hours)
mx, my = hours.mean(), score.mean()
Sxx_ = ((hours - mx) ** 2).sum()
Sxy_ = ((hours - mx) * (score - my)).sum()

bb1 = Sxy_ / Sxx_
bb0 = my - bb1 * mx
fit_ = bb0 + bb1 * hours
res_ = score - fit_
sse_ = (res_ ** 2).sum()
sst_ = ((score - my) ** 2).sum()
sigma_ = np.sqrt(sse_ / (nn - 2))
se_ = sigma_ / np.sqrt(Sxx_)
tc_ = stats.t(nn - 2).ppf(0.975)

print(f"slope     = {Sxy_:.2f} / {Sxx_:.2f} = {bb1:.4f}")
print(f"intercept = {my:.2f} - {bb1:.4f} * {mx:.2f} = {bb0:.4f}")
print(f"R^2       = 1 - {sse_:.2f}/{sst_:.2f} = {1 - sse_/sst_:.4f}")
print(f"RMSE      = {np.sqrt(sse_/nn):.4f} marks")
print(f"SE(slope) = {se_:.4f}")
print(f"95% CI    = [{bb1 - tc_*se_:.4f}, {bb1 + tc_*se_:.4f}]")
print(f"t         = {bb1/se_:.3f}, p = {2*stats.t(nn-2).sf(abs(bb1/se_)):.2e}")

chk = stats.linregress(hours, score)
print(f"\nscipy: slope={chk.slope:.4f}, intercept={chk.intercept:.4f}, "
      f"r^2={chk.rvalue**2:.4f}, stderr={chk.stderr:.4f}")
print(f"\nInterpretation: each extra hour of study is associated with about "
      f"{bb1:.2f} more marks.")

**Exercise 2.** Fit a multiple regression on the bundled diabetes dataset predicting disease
progression from BMI, blood pressure and the `s5` blood serum measure.
(a) Report the coefficient table and interpret the BMI coefficient.
(b) Which predictors are significant?
(c) Compare $R^2$ and adjusted $R^2$ with a model using all ten features.
(d) Produce the diagnostic plots and comment.

In [ ]:
# --- Solution 2 -------------------------------------------------------------
from sklearn.datasets import load_diabetes

dia = load_diabetes(as_frame=True)
D = dia.frame
sel = ["bmi", "bp", "s5"]

tab2, st2, res2, fit2 = ols_summary(D[sel].to_numpy(), D["target"].to_numpy(), sel)
print(tab2.round(3).to_string(index=False))
print(f"\nn={st2['n']}  R^2={st2['R2']:.4f}  adjusted R^2={st2['R2_adj']:.4f}  "
      f"RMSE={st2['RMSE']:.2f}")
print("\n(a) The features are standardised, so the BMI coefficient "
      f"({tab2.loc[1,'estimate']:.1f}) is the")
print("    expected change in the progression score per one-unit (one-sd-ish) rise in BMI,")
print("    holding blood pressure and s5 fixed.")
print(f"(b) Significant at 5%: "
      f"{[t for t, p in zip(tab2.term, tab2.p_value) if p < 0.05 and t != 'intercept']}")

allf = [c for c in D.columns if c != "target"]
tab_all, st_all, _, _ = ols_summary(D[allf].to_numpy(), D["target"].to_numpy(), allf)
print(f"\n(c) 3 features : R^2={st2['R2']:.4f}  adj={st2['R2_adj']:.4f}")
print(f"    10 features: R^2={st_all['R2']:.4f}  adj={st_all['R2_adj']:.4f}")
print("    R^2 rises with all ten, and adjusted R^2 rises too -- so the extra features")
print("    are earning their keep here, unlike the pure-noise columns in section 9.4.")

In [ ]:
# (d) diagnostics for the three-predictor model
diagnostic_plots(res2, fit2, "Diabetes model diagnostics")
print(f"Shapiro-Wilk p = {stats.shapiro(res2).pvalue:.4f}")
print(f"Durbin-Watson  = {durbin_watson(res2):.3f}")
print("\nComment: residuals are roughly symmetric and the Q-Q plot is close to straight,")
print("but the residuals-vs-fitted panel shows a mild funnel -- variance grows for")
print("patients with worse predicted progression. The point estimates are usable;")
print("the standard errors are mildly optimistic.")

**Exercise 3.** A colleague reports "$R^2 = 0.91$, so the model is excellent". Construct a
dataset where $R^2 = 0.91$ but the model is badly wrong, and explain what they should have
looked at instead.

In [ ]:
# --- Solution 3 -------------------------------------------------------------
xs3 = np.linspace(0, 10, 120)
y3 = 5 + 3*xs3 + 1.4*(xs3 - 5)**2 + rng.normal(0, 3, 120)     # genuinely quadratic

lin = LinearRegression().fit(xs3.reshape(-1, 1), y3)
pred_lin = lin.predict(xs3.reshape(-1, 1))
print(f"Straight-line fit to curved data: R^2 = {r2_score(y3, pred_lin):.4f}")

X_quad = np.column_stack([xs3, xs3**2])
quad = LinearRegression().fit(X_quad, y3)
print(f"Quadratic fit                   : R^2 = {r2_score(y3, quad.predict(X_quad)):.4f}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
ax[0].scatter(xs3, y3, s=16, alpha=0.7, color="steelblue")
ax[0].plot(xs3, pred_lin, color="crimson", lw=2, label="linear fit")
ax[0].plot(xs3, quad.predict(X_quad), color="seagreen", lw=2, label="quadratic fit")
ax[0].legend(fontsize=8); ax[0].set_title(f"High R^2, wrong shape")
ax[1].scatter(pred_lin, y3 - pred_lin, s=16, color="steelblue")
ax[1].axhline(0, color="crimson")
ax[1].set_title("The residual plot gives the game away")
ax[1].set_xlabel("fitted"); ax[1].set_ylabel("residual")
plt.tight_layout(); plt.show()

print()
print("What they should have checked:")
print("  1. The residuals-vs-fitted plot -- the U shape screams 'missing curvature'")
print("  2. RMSE in the units of y, against what accuracy the decision needs")
print("  3. Performance on data the model has NOT seen (Notebook 10)")
print("  4. Whether predictions are systematically wrong in a region that matters")
print(f"\nRMSE linear = {np.sqrt(mean_squared_error(y3, pred_lin)):.2f}, "
      f"quadratic = {np.sqrt(mean_squared_error(y3, quad.predict(X_quad))):.2f}")

**Exercise 4 (challenge).** Demonstrate **omitted variable bias** and then show that
including the omitted variable fixes it. Simulate: `experience` affects both `training_hours`
and `salary`, while `training_hours` has only a small true effect on salary. Show that a
regression of salary on training hours alone badly overstates the effect of training, and
quantify the bias formula
$\text{bias} = \beta_{\text{omitted}} \times \dfrac{\operatorname{Cov}(x_{\text{included}}, x_{\text{omitted}})}{\operatorname{Var}(x_{\text{included}})}$.

In [ ]:
# --- Solution 4 -------------------------------------------------------------
N4 = 2_000
experience = rng.uniform(0, 25, N4)
training   = 8 + 1.4 * experience + rng.normal(0, 4, N4)      # experienced staff train more
TRUE_TRAIN, TRUE_EXP = 400.0, 9_000.0
salary = 250_000 + TRUE_TRAIN*training + TRUE_EXP*experience + rng.normal(0, 20_000, N4)

t_naive, _, _, _ = ols_summary(training.reshape(-1, 1), salary, ["training_hours"])
t_full,  _, _, _ = ols_summary(np.column_stack([training, experience]), salary,
                               ["training_hours", "experience"])

print("Regression on training hours ALONE (experience omitted):")
print(t_naive.round(1).to_string(index=False))
print("\nRegression including experience:")
print(t_full.round(1).to_string(index=False))

naive_coef = t_naive.loc[1, "estimate"]
full_coef  = t_full.loc[1, "estimate"]
predicted_bias = TRUE_EXP * np.cov(training, experience, ddof=1)[0, 1] / training.var(ddof=1)

print(f"\nTrue effect of training      : {TRUE_TRAIN:,.0f} per hour")
print(f"Estimate controlling for exp : {full_coef:,.0f}  (close to the truth)")
print(f"Naive estimate               : {naive_coef:,.0f}  (inflated {naive_coef/TRUE_TRAIN:.1f}x)")
print(f"Observed bias                : {naive_coef - TRUE_TRAIN:,.0f}")
print(f"Bias formula prediction      : {predicted_bias:,.0f}")
print()
print("The naive model credits training with the salary premium that experience earns.")
print("A manager acting on it would over-invest in training courses.")
print("Omitted variable bias is the regression form of the confounding you met in")
print("Notebook 5 -- and no amount of data fixes it. Only a better model, or")
print("randomisation, does.")

---
## Summary

| Concept | Formula / key point |
|---|---|
| Model | $y = \beta_0 + \beta_1 x + \varepsilon$ |
| OLS slope | $\hat\beta_1 = s_{xy}/s_x^2 = r\,s_y/s_x$ |
| OLS intercept | $\hat\beta_0 = \bar{y} - \hat\beta_1\bar{x}$ |
| Matrix form | $\hat{\boldsymbol\beta} = (X^\top X)^{-1}X^\top \mathbf{y}$ |
| Decomposition | SST = SSR + SSE |
| $R^2$ | $1 - \text{SSE}/\text{SST}$; equals $r^2$ for one predictor |
| Adjusted $R^2$ | Penalises extra predictors — use it for comparisons |
| RMSE / MAE | Error in the units of $y$; always report one |
| SE of slope | $\hat\sigma/\sqrt{S_{xx}}$ — smaller with more, less noisy, more spread-out $x$ |
| Inference | $t = \hat\beta_j/\operatorname{SE}$, $df = n-k-1$ |
| Multiple regression | $\beta_j$ = effect of $x_j$ **holding others fixed** |
| Dummy variables | $L$ levels → $L-1$ columns; one is the reference |
| Assumptions | **LINE** + no perfect multicollinearity |
| Influence | Cook's $D > 4/n$; investigate, do not just delete |
| Omitted variable bias | Leaving out a confounder biases the coefficients you keep |

**Next up:** [Notebook 10 — Train-Test Split](10.%20Train-Test%20Split.ipynb). Everything so
far measured fit on the *same* data used to fit. That number is optimistic, and the next
three notebooks are about fixing it.